# Customer Service Chatbot with LangGraph

This notebook implements an intelligent customer service chatbot that uses LangGraph to route conversations based on user intent. The system manages customer support tickets through a state machine architecture that can:

- Handle positive feedback and mark tickets as resolved
- Process negative feedback and update ticket notes
- Answer generic queries about ticket status
- Generate new tickets from customer complaints

## Architecture Overview

The chatbot uses a graph-based workflow where:
1. User messages are classified into one of four categories
2. Messages are routed to specialized resolver functions
3. Each resolver updates ticket data in a CSV file
4. A final response is generated and returned to the user

## Prerequisites

Before running this notebook, ensure you have:
- A valid Groq API key
- The test_data.csv file in the same directory
- Installed all dependencies (see requirements.txt)

In [ ]:
!pip install pandas
!pip install -U langchain-chroma
!pip install -U langchain_huggingface
!pip install -U sentence-transformers langchain.tools
!pip install langgraph
!pip install grandalf
!pip install langchain_graph groq langchain_groq
!pip install langgraph langchain_groq langchain_chroma langchain_huggingface pydantic typing-extensions

## State Management and LLM Configuration

This section defines the core data structures and initializes the language model that powers the chatbot.

### State Dictionary
The `State` TypedDict tracks:
- **messages**: Conversation history with support for message aggregation
- **user_input**: The current user query
- **message_type**: Classification result (generic_query_resolver, positive_feedback_resolver, etc.)
- **ticket_number**: The support ticket ID being discussed

### LLM Initialization
The chatbot uses ChatGroq with the gpt-oss-120b model. This model handles:
- Message classification
- Response generation
- Ticket summarization

Note: Replace `<api_key>` with your actual Groq API key.

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_groq import ChatGroq
from pydantic import BaseModel, Field
from typing import Annotated
from typing_extensions import TypedDict

class State(TypedDict):
    messages: Annotated[list[str], add_messages]
    user_input: str
    message_type: str
    ticket_number: int


from langchain_groq import ChatGroq
llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0, api_key="<api_key>")

## Message Classifier Schema

This Pydantic model defines the four possible message types the system can handle:

1. **generic_query_resolver**: User wants ticket status information
2. **positive_feedback_resolver**: User is satisfied with the resolution
3. **negative_feedback_resolver**: User is still experiencing issues
4. **new_ticket_generator**: User is reporting a new problem without an existing ticket

The classifier is used with structured output to ensure the LLM returns one of these exact values.

In [ ]:
from typing import Literal


class MessageClassifier(BaseModel):
    message_type: Literal["generic_query_resolver", "positive_feedback_resolver", "negative_feedback_resolver", "new_ticket_generator"] = Field(description='message type')


## Positive Feedback Resolver

This function handles cases where the user confirms their issue has been resolved.

### Process Flow:
1. Reads the current ticket data from test_data.csv
2. Updates the ticket status to "Resolved"
3. Generates a friendly acknowledgment message thanking the user
4. Returns the response with the updated ticket number

### Database Interaction:
Uses pandas to read, update, and write the CSV file that serves as the ticket database.

In [ ]:
import os
import pandas as pd

def positive_feedback_resolver(state: State) -> str:
    '''
    Use this tool to provide a friendly response expressing gratitude towards the user upon successful resolution of their ticket
    '''
    print('Positive Feedback resolver invoked')

    ticket_number = state.get('ticket_number')

    system_prompt = f'\
                    Respond to the positive feedback received by the user. \
                    Greet them with a response indicating the pleasure of interacting with them and conducting business with them\
                    and indicate that ticket_number {ticket_number} has been marked as resolved'

    df = pd.read_csv('test_data.csv')

    df.loc[df['ticket_number']==state['ticket_number'][0], 'status']='Resolved'
    df.to_csv('test_data.csv', index=False)

    positive_feedback_prompt = [
        {
            "role": "system",
            "content": system_prompt
        },
        {
            "role": "user",
            "content": state['messages'][0].content
        }
    ]

    result = llm.invoke(positive_feedback_prompt)
    return {"messages": [{"content": result.content, "role": "assistant"}], "ticket_number":ticket_number}


## New Ticket Generator

Creates a new support ticket when the user reports a problem and no ticket number exists.

### Process Flow:
1. Generates a new ticket number (increments from the last ticket in the database)
2. Uses the LLM to summarize the user's issue into a concise description
3. Creates a new row in the ticket database with status "Unresolved"
4. Responds to the user with the new ticket number and summary

### Dual LLM Calls:
This function makes two LLM calls:
- First call: Summarizes the user feedback into under 50 words
- Second call: Generates a friendly response acknowledging the issue

In [ ]:
import os
import pandas as pd

def new_ticket_generator(state: State) -> str:
    '''
    Use this tool to provide a friendly response expressing gratitude towards the user upon successful resolution of their ticket
    '''
    print('New ticket resolver invoked')

    df = pd.read_csv('test_data.csv')
    new_ticket_number = df['ticket_number'].astype(int).iloc[-1] + 1


    # We will need 2 llm calls here
    # 1. Summarize and update the existing status
    # 2. Provide a response to the user

    summarizer_system_prompt = """
                    Summarize {user_feedback} to create a new summary for the ticket
                    The summary should be less than 50 words
                    """

    df = pd.read_csv('test_data.csv')
    user_feedback=state['messages'][0].content

    summarizer_prompt = [
        {
            "role": "system",
            "content": summarizer_system_prompt
        },
        {
            "role": "user",
            "content": user_feedback
        }
    ]

    summary = llm.invoke(summarizer_prompt)

    system_prompt = f'\
                    Respond to the feedback received by the user. \
                    Greet them with a response indicating the pleasure of interacting with them and conducting business with them\
                    Analyze the feedback, if it is positive, provide a friendly greeting expressing gratitude for the feedback\
                    if feedback is negative\
                    and indicate that ticket_number:{new_ticket_number} has been created with this summary: {summary}\
                    if feedback cannot be classified, give a friendly message stating you are not able to process the query and ask how you can help'
    df.loc[len(df)] = [new_ticket_number, 'Unresolved', summary.content.replace("\n", "")]

    df.to_csv('test_data.csv', index=False)

    negative_feedback_prompt = [
        {
            "role": "system",
            "content": system_prompt
        },
        {
            "role": "user",
            "content": state['messages'][0].content
        }
    ]

    result = llm.invoke(negative_feedback_prompt)
    return {"messages": [{"content": result.content, "role": "assistant"}], "ticket_number":new_ticket_number}


## Generic Query Resolver

Handles standard inquiries about existing ticket status.

### Process Flow:
1. Looks up the ticket in the database using the provided ticket number
2. Retrieves current status and notes
3. Generates a friendly summary response
4. If no ticket is found, informs the user politely

### Use Cases:
- "What's the status of my ticket?"
- "Can you check on ticket 1064?"
- "Has my issue been resolved?"

In [ ]:
from collections import abc
import os
import pandas as pd

def generic_query_resolver(state: State) -> str:
    '''
    Use this tool to provide a friendly current status of ticket raised by user and express gratitude for their correspondence
    '''
    print('Generic Query resolver invoked')
    df = pd.read_csv('test_data.csv')
    ticket_number = state.get('ticket_number')
    current_row = df[df['ticket_number']==ticket_number]
    current_status = current_row['status']
    notes = current_row['notes']


    system_prompt = f'Greet with a friendly message acknowledging the user input For the ticket_number: {ticket_number}, current_status: {current_status} and notes: {notes} \
                    Generate a summary and provide a response\
                    if any of the values are missing simply provide a friendly message indicating no recordws for the ticket_number: {ticket_number} could be found'


    generic_query_prompt = [
        {
            "role": "system",
            "content": system_prompt
        },
        {
            "role": "user",
            "content": state['messages'][0].content
        }
    ]

    result = llm.invoke(generic_query_prompt)
    return {"messages": [{"content": result.content, "role": "assistant"}], "ticket_number":ticket_number}


## Negative Feedback Resolver

Processes ongoing issues when the user reports their problem is not resolved.

### Process Flow:
1. Retrieves existing ticket notes from the database
2. Combines user feedback with current notes
3. Uses the LLM to create an updated summary (under 50 words)
4. Updates ticket status to "Unresolved"
5. Stores the new summary in the database
6. Generates an apologetic response indicating continued support

### Dual LLM Calls:
- First call: Summarizes combined feedback and notes
- Second call: Generates empathetic response to the user

In [ ]:
import os
import pandas as pd

def negative_feedback_resolver(state: State) -> str:
    '''
    Use this tool to provide a friendly response expressing gratitude towards the user upon successful resolution of their ticket
    '''
    print('Negative Feedback resolver invoked')

    # We will need 2 llm calls here
    # 1. Summarize and update the existing status
    # 2. Provide a response to the user

    summarizer_system_prompt = """
                    Summarize {user_feedback} and {current_notes} to create a new summary for the status of the ticket
                    The summary should be less than 50 words
                    """

    df = pd.read_csv('test_data.csv')
    user_feedback=state['messages'][0].content
    ticket_number = state.get('ticket_number')
    current_notes = df[df['ticket_number']==ticket_number]['notes'].item()
    summarizer_prompt = [
        {
            "role": "system",
            "content": summarizer_system_prompt
        },
        {
            "role": "user",
            "content": f'user_feedback: {user_feedback} current_notes: {current_notes}'
        }
    ]

    summarized_notes = llm.invoke(summarizer_prompt).content.replace("\n", "")
    df.loc[df['ticket_number']==ticket_number, 'status']='Unresolved'
    df.loc[df['ticket_number']==ticket_number, 'notes']=summarized_notes
    df.to_csv('test_data.csv', index=False)
    system_prompt = f'\
                    Analyze the negative feedback provider by customer in {user_feedback}, provide the summary you captured in the {summarized_notes}\
                    and provide a apologetic response indicating you are working earnestly to address their issue\
                    '
    negative_feedback_prompt = [
        {
            "role": "system",
            "content": system_prompt
        }
    ]

    result = llm.invoke(negative_feedback_prompt)
    return {"messages": [{"content": result.content, "role": "assistant"}], "ticket_number":state.get("ticket_number")}


## Router Function

The router is the entry point of the chatbot workflow. It classifies incoming messages and determines which resolver to use.

### Classification Logic:
- **positive_feedback_resolver**: User expresses satisfaction ("Thanks for fixing this!")
- **negative_feedback_resolver**: User still has problems ("It's still broken")
- **generic_query_resolver**: User requests ticket information
- **new_ticket_generator**: No ticket number provided and user describes a new issue

### Structured Output:
Uses the MessageClassifier Pydantic model to ensure the LLM returns a valid message type that matches one of the graph nodes.

In [ ]:
def router_message(state: State):
    user_input = state['messages'][0].content
    classifier_llm = llm.with_structured_output(MessageClassifier)
    ticket_number = state.get("ticket_number")
    system_prompt = """
                    Classify user message as 'positive_feedback_resolver', 'negative_feedback_resolver' or 'generic_query_resolver' based on the criteria below
                    'positive_feedback_resolver': if the user has provided input that indicates they are satisfied or happy with the resolution of their ticket, eg:  “Thanks for resolving my issue!"
                    'negative_feedback_resolver': if the user is still dissatisfied by the resolution Eg: “My problem still isn't fixed”
                    'generic_query_resolver': if user needs generic information regarding existing tickets or if they are seeking resolution for a brand new issue
                    'new_ticket_generator': if there is {ticket_number} is undefined/null and user is providing a brand new ticket
                    """

    final_prompt = [
        {
            "role": "system",
            "content": system_prompt
        },
        {
            "role": "user",
            "content": user_input
        }
    ]


    result = classifier_llm.invoke(final_prompt)
    return {"messages": [{"content": result.message_type, "role":"assistant"}], "user_input":user_input, "message_type":result.message_type, "ticket_number":ticket_number}

## Final Response Formatter

This function generates the final response to the user after a resolver has processed the request.

### Process:
1. Extracts the user's original query
2. Retrieves the context from the last message (resolver's output)
3. Combines query, ticket number, and context
4. Uses the LLM to generate a coherent final response
5. Handles edge cases where context doesn't align with the query

### Safety Check:
If the context is misaligned with the user query, the response will be "No Context found" to avoid hallucination.

In [ ]:
def final_response(state: State):
    user_input = state["user_input"]
    context = state["messages"][-1].content # the last message is the final response
    ticket_number = state["ticket_number"]

    final_prompt = f'''
                    refer to the query and context below to build the response
                    if context is not aligned with the user query, the response must be 'No Context found'. Do not use
                    any additional context
                    user_query : {user_input}
                    ticket_number: {ticket_number}
                    context: {context}
                    '''
    result = llm.invoke(final_prompt)

    return {"messages": [{"content":result.content, "role":"assistant"}]}


## Graph Construction

This section builds the state machine that orchestrates the conversation flow.

### Graph Structure:
- **Nodes**: Each resolver function is a node, plus the router and final_response
- **Edges**: Define the flow between nodes
  - START → router (entry point)
  - router → [one of four resolvers] (conditional based on message_type)
  - All resolvers → final_response (converge to output)

### Visualization:
The `print_ascii()` call outputs an ASCII diagram showing the complete workflow graph.

In [ ]:
graph_builder = StateGraph(State)
graph_builder.add_node("router", router_message)
graph_builder.add_node("positive_feedback_resolver", positive_feedback_resolver)
graph_builder.add_node("negative_feedback_resolver", negative_feedback_resolver)
graph_builder.add_node("generic_query_resolver", generic_query_resolver)
graph_builder.add_node("new_ticket_generator", new_ticket_generator)

graph_builder.add_node("final_response", final_response)

graph_builder.add_edge(START, "router")
graph_builder.add_conditional_edges("router", lambda state: state.get("message_type"), {"positive_feedback_resolver": "positive_feedback_resolver", "negative_feedback_resolver": "negative_feedback_resolver", "generic_query_resolver": "generic_query_resolver", "new_ticket_generator": "new_ticket_generator"})
graph_builder.add_edge("positive_feedback_resolver", "final_response")
graph_builder.add_edge("negative_feedback_resolver", "final_response")
graph_builder.add_edge("new_ticket_generator", "final_response")
graph_builder.add_edge("generic_query_resolver", "final_response")

graph = graph_builder.compile()

graph.get_graph().print_ascii()


## Test Cases

The following cells demonstrate the chatbot handling different types of customer interactions:

### Test 1: Negative Feedback
User expresses dissatisfaction with existing ticket 1064. The system should:
- Route to negative_feedback_resolver
- Update ticket notes
- Generate an apologetic response

### Test 2: Status Query
User asks about ticket 1006 status. The system should:
- Route to generic_query_resolver
- Look up the ticket
- Return current status and notes

### Test 3: New Ticket
User reports a new issue without providing a ticket number. The system should:
- Route to new_ticket_generator
- Create a new ticket
- Return the new ticket number and summary

In [ ]:
result2 = graph.invoke({'messages':[{"role": "user", "content":"I am extremely displeased with your service"}], 'ticket_number':1064})
result2['messages']

In [ ]:
result3 = graph.invoke({'messages':[{"role": "user", "content":"what is the current status of this ticket"}], 'ticket_number':1006})
result3

In [ ]:
result4 = graph.invoke({'messages':[{"role": "user", "content":"My diecast car was never delivered"}]})
result4['messages'][-1]